In [2]:
pip install gnews

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6105 sha256=e08db15e4e1bded74fdcde4fa46fc715714d3494dcf03cbe9bbdf5ea5244f597
  Stored in directory: c:\users\hp\appdata\local\pip\cache\wheels\f0\69\93\a47e9d621be168e9e33c7ce60524393c0b92ae83cf6c6e89c5
Successfully built sgmllib3k

   ---------- ----------------------------- 1/4 [feedparser]
   ---------- ----------------------------- 1/4 [feedparser]
   ---------- ----------------------------- 1/4 [feedparser]
   ---------- ----------------------------- 1/4 [feedparser]
   ---------- ----------------------------- 1/4 [feedparser]
   ---------- ----------------------------- 1/4 [feedparser]
   -------------------- ------------------- 2/4 [dnspython]
   -------------------- ------------------- 2/4 [dnspython]
   -------------------- ------------------- 2/4 [dnspython]
   -------------------- -----------------

  DEPRECATION: Building 'sgmllib3k' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'sgmllib3k'. Discussion can be found at https://github.com/pypa/pip/issues/6334

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
import numpy as np
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from gnews import GNews
from datetime import datetime, timedelta

In [2]:
stocks = {
    "RELIANCE": "Reliance Industries",
    "HDFCBANK": "HDFC Bank",
    "INFY": "Infosys",
    "M&M": "Mahindra and Mahindra",
    "BHARTIARTL": "Bharti Airtel",
    "HUL": "Hindustan Unilever"
}

In [3]:
from datetime import datetime, timedelta
from gnews import GNews
import pandas as pd
import os
import json

# Initialize GNews
google_news = GNews(language="en", country="IN", max_results=5)

start_date = datetime(2019, 2, 1)
end_date   = datetime.today()

# Create folders
os.makedirs("company_news", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

# Example:
# stocks = {"RELIANCE": "Reliance Industries", "TCS": "Tata Consultancy Services"}

for ticker, company in stocks.items():
    
    print(f"\n==============================")
    print(f"Processing {company}")
    print(f"==============================")

    csv_path = f"company_news/{ticker}_weekly_news.csv"
    checkpoint_path = f"checkpoints/{ticker}_checkpoint.json"

    # 🔹 Load checkpoint if exists
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            checkpoint_data = json.load(f)
            current = datetime.strptime(checkpoint_data["last_processed_week"], "%Y-%m-%d")
        print(f"Resuming from: {current.strftime('%Y-%m-%d')}")
    else:
        current = start_date
        print("Starting fresh from:", start_date.strftime("%Y-%m-%d"))

    while current <= end_date:
        
        next_week = current + timedelta(days=7)
        print("\nProcessing week:", current.strftime("%Y-%m-%d"))

        google_news.start_date = current
        google_news.end_date   = next_week

        query = f"{company} stock OR shares OR earnings OR results OR NSE OR BSE"

        try:
            articles = google_news.get_news(query)
        except Exception as e:
            print("Error:", e)
            articles = []

        # 🔹 NEW LINE → Print number of articles found
        print(f"Articles found: {len(articles)}")

        weekly_rows = []

        for article in articles:
            weekly_rows.append({
                "week_start": current.strftime("%Y-%m-%d"),
                "ticker": ticker,
                "company": company,
                "title": article.get("title"),
                "published_date": article.get("published date"),
                "url": article.get("url")
            })

        # 🔹 Append to CSV immediately (checkpoint-safe)
        if weekly_rows:
            weekly_df = pd.DataFrame(weekly_rows)

            if os.path.exists(csv_path):
                weekly_df.to_csv(csv_path, mode='a', header=False, index=False)
            else:
                weekly_df.to_csv(csv_path, index=False)

        # 🔹 Save checkpoint AFTER week finishes
        with open(checkpoint_path, "w") as f:
            json.dump({"last_processed_week": next_week.strftime("%Y-%m-%d")}, f)

        print(f"Saved checkpoint at {next_week.strftime('%Y-%m-%d')}")

        current = next_week

    print(f"\nFinished {company}")


Processing Reliance Industries
Resuming from: 2026-02-27

Finished Reliance Industries

Processing HDFC Bank
Resuming from: 2026-02-27

Finished HDFC Bank

Processing Infosys
Resuming from: 2026-02-27

Finished Infosys

Processing Mahindra and Mahindra
Resuming from: 2026-02-27

Finished Mahindra and Mahindra

Processing Bharti Airtel
Resuming from: 2026-02-27

Finished Bharti Airtel

Processing Hindustan Unilever
Resuming from: 2026-02-27

Finished Hindustan Unilever
